# 🧬 BOI Sentinel AI — Bytecode Visualization CNN (v1)

**Purpose:** Train a CNN to classify Android malware by converting DREBIN feature vectors into grayscale images.  
**Dataset:** DREBIN-215 (auto-downloaded from Kaggle)  
**Output:** `cnn_malware_model.tflite` + `cnn_metadata.json`

In [ ]:
!pip install -q kagglehub tensorflow scikit-learn matplotlib seaborn numpy pillow

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score, roc_curve

print(f'TF: {tf.__version__}, GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
import kagglehub
path = kagglehub.dataset_download('shashwatwork/android-malware-dataset-for-machine-learning')
print(f'Downloaded to: {path}')
for f in sorted(os.listdir(path)):
    fpath = os.path.join(path, f)
    if f.endswith('.csv'):
        temp = pd.read_csv(fpath, nrows=2)
        print(f'  {f}: {temp.shape[1]} cols')

In [ ]:
# Load the 216-column DREBIN dataset
CSV_PATH = os.path.join(path, 'drebin-215-dataset-5560malware-9476-benign.csv')
df = pd.read_csv(CSV_PATH)
print(f'Shape: {df.shape}')

X_raw = df.iloc[:, :-1].apply(pd.to_numeric, errors='coerce').fillna(0)
y_raw = df.iloc[:, -1]
y = y_raw.map({'S': 1, 'B': 0}).fillna(1).astype(int)

print(f'Benign: {(y==0).sum()}, Malware: {(y==1).sum()}')

In [ ]:
# ── Convert feature vectors → grayscale images ──
# Each 215-dim binary vector becomes a 64x64 image (4096 pixels, padded with 0)
FEAT_IMG_SIZE = 64

def features_to_image(fv, img_size=FEAT_IMG_SIZE):
    total = img_size * img_size
    scaled = (fv * 255).astype(np.uint8)
    if len(scaled) < total:
        scaled = np.pad(scaled, (0, total - len(scaled)))
    else:
        scaled = scaled[:total]
    return scaled.reshape(img_size, img_size)

X_features = X_raw.values.astype(np.float32)
images = np.array([features_to_image(row) for row in X_features])
print(f'Images: {images.shape}, Range: [{images.min()}, {images.max()}]')

In [ ]:
# Visualize samples
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Benign (top) vs Malware (bottom)', fontsize=14)
for i, idx in enumerate(np.where(y.values==0)[0][:5]):
    axes[0,i].imshow(images[idx], cmap='gray'); axes[0,i].axis('off'); axes[0,i].set_title(f'Benign')
for i, idx in enumerate(np.where(y.values==1)[0][:5]):
    axes[1,i].imshow(images[idx], cmap='gray'); axes[1,i].axis('off'); axes[1,i].set_title(f'Malware')
plt.tight_layout(); plt.show()

In [ ]:
# Prepare data
X = images.astype(np.float32) / 255.0
X = X.reshape(-1, FEAT_IMG_SIZE, FEAT_IMG_SIZE, 1)
y_arr = y.values.astype(np.int32)

X_train, X_test, y_train, y_test = train_test_split(X, y_arr, test_size=0.2, random_state=42, stratify=y_arr)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)

n = len(y_train)
class_weights = {0: n/(2*sum(y_train==0)), 1: n/(2*sum(y_train==1))}
print(f'Train: {len(y_train)}, Val: {len(y_val)}, Test: {len(y_test)}')
print(f'Class weights: {class_weights}')

In [ ]:
# Build CNN
model = models.Sequential([
    layers.Conv2D(32, 3, padding='same', input_shape=(FEAT_IMG_SIZE, FEAT_IMG_SIZE, 1)),
    layers.BatchNormalization(), layers.ReLU(),
    layers.Conv2D(32, 3, padding='same'),
    layers.BatchNormalization(), layers.ReLU(),
    layers.MaxPooling2D(2), layers.Dropout(0.25),

    layers.Conv2D(64, 3, padding='same'),
    layers.BatchNormalization(), layers.ReLU(),
    layers.Conv2D(64, 3, padding='same'),
    layers.BatchNormalization(), layers.ReLU(),
    layers.MaxPooling2D(2), layers.Dropout(0.25),

    layers.Conv2D(128, 3, padding='same'),
    layers.BatchNormalization(), layers.ReLU(),
    layers.GlobalAveragePooling2D(),

    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')],
)
model.summary()

In [ ]:
# Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=64,
    class_weight=class_weights,
    callbacks=[
        EarlyStopping(monitor='val_auc', patience=10, restore_best_weights=True, mode='max', verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    ],
    verbose=1,
)

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, key, title in zip(axes, ['loss','accuracy','auc'], ['Loss','Accuracy','AUC']):
    ax.plot(history.history[key], label='Train', lw=2)
    ax.plot(history.history[f'val_{key}'], label='Val', lw=2)
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Evaluate
CLASS_NAMES = ['Benign', 'Malware']
y_pred_proba = model.predict(X_test, verbose=0).flatten()
y_pred = (y_pred_proba >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred)

print(f'Accuracy: {acc:.4f}  |  AUC: {auc:.4f}  |  F1: {f1:.4f}')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
axes[0].set_title(f'Confusion Matrix (Acc={acc:.3f})'); axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, 'b-', lw=2, label=f'AUC={auc:.4f}')
axes[1].plot([0,1],[0,1],'r--',alpha=0.5); axes[1].set_title('ROC'); axes[1].legend(); axes[1].grid(True,alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Export
OUTPUT_DIR = 'trained_cnn_model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Keras
model.save(os.path.join(OUTPUT_DIR, 'cnn_malware_model.keras'))

# TFLite (quantized)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
tflite_path = os.path.join(OUTPUT_DIR, 'cnn_malware_model.tflite')
with open(tflite_path, 'wb') as f: f.write(tflite_model)

# Metadata
meta = {'model_type':'CNN','version':'v1','input_shape':[FEAT_IMG_SIZE,FEAT_IMG_SIZE,1],
        'class_names':CLASS_NAMES,'feature_count':int(X_raw.shape[1]),
        'test_accuracy':round(float(acc),4),'test_auc':round(float(auc),4),'test_f1':round(float(f1),4),
        'feature_names':list(X_raw.columns)}
with open(os.path.join(OUTPUT_DIR, 'cnn_metadata.json'), 'w') as f: json.dump(meta, f, indent=2)

# Verify
interp = tf.lite.Interpreter(model_path=tflite_path); interp.allocate_tensors()
inp = interp.get_input_details(); out = interp.get_output_details()
interp.set_tensor(inp[0]['index'], X_test[0:1].astype(np.float32))
interp.invoke()
print(f'TFLite verify: p(malware)={float(interp.get_tensor(out[0]["index"])[0][0]):.4f}')

for f in os.listdir(OUTPUT_DIR):
    sz = os.path.getsize(os.path.join(OUTPUT_DIR,f))
    print(f'  ✅ {f} ({sz/1024:.1f} KB)')

print(f'\n➡️ Copy trained_cnn_model/ files to services/risk-scoring/models/')

In [ ]:
# Download from Colab
try:
    from google.colab import files
    for f in os.listdir(OUTPUT_DIR): files.download(os.path.join(OUTPUT_DIR, f))
    print('Downloaded!')
except ImportError:
    print(f'Files saved in: {OUTPUT_DIR}')